# 2026 지옥캠프: 6공학관 vs IT관 분류 — 베이스라인 (Small CNN)

약 40만 파라미터의 소형 CNN으로, 끝까지 실행하면 제출용 CSV와 모델 패키지가 생성되고 그럴듯한 점수가 나오는 출발점입니다.

- 작성 환경: Google Colab (CPU만으로도 동작) / 로컬 Python 3.10+, PyTorch 2.x
- GPU가 있으면 자동으로 사용하지만 **필수는 아닙니다**

## 대회 규칙 요약 (필독)

| 항목 | 내용 |
|---|---|
| 과제 | 사진 → **6공학관(0) / IT관(1)** 이진 분류 |
| 참가 | 개인전 |
| 기간 | 2026-11-07(토) ~ 11-08(일) |
| 지표 | **Log Loss** (낮을수록 우수, 확률을 제출해야 함) |
| 모델 제약 | **총 파라미터 20M 이하** (앙상블·사전학습 백본 합산) |
| 런타임 | **Colab CPU 기준** 테스트 500장 10분 이내 · 실행 중 다운로드 금지 (사전학습 가중치는 최종 파일에 포함) |
| 금지 | 외부 이미지 사용, 테스트 이미지를 학습에 사용(pseudo-labeling 포함) |
| 제출 | 예측 CSV **10분에 1회 · 하루 최대 50회** (공개 리더보드) + 마감 시 최종 파일 1개 (형식 자유) |
| 순위 | 테스트 500장 중 Public 250장 = 공개 리더보드, **Private 250장 = 최종 순위** (어느 장이 Private 인지는 비공개) |


## 실행 전 사전 준비

### 모듈
Colab은 필요한 모듈이 모두 설치되어 있습니다. 로컬이라면:
```shell
# 본인 환경(CPU/CUDA)에 맞는 PyTorch 설치: https://pytorch.org/get-started/locally/
pip3 install torch torchvision
pip3 install -U scikit-learn matplotlib tqdm ipywidgets
```

### 데이터 파일
운영진이 배포한 데이터를 노트북과 같은 폴더에 압축해제해 주세요 (Colab 은 `/content`).
```
📁root
 ├─📁train
 │  ├─📁6e      ← 6공학관 (label 0)
 │  └─📁it      ← IT관   (label 1)
 ├─📁test       ← 테스트 500장 (Public 250 + Private 250)
 └─📜baseline.ipynb
```


## 모듈 불러오기 & 시드 고정

재현성을 위해 모든 난수 시드를 고정합니다. 시드가 고정되어 있어야 "내가 바꾼 것 때문에 점수가 오른 것인지, 운이 좋았던 것인지"를 구분할 수 있습니다.

In [ ]:
import os
import random
import shutil
import subprocess
import time

import numpy as np
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, Dataset, Subset
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from PIL import Image
# tqdm.auto 는 DataLoader 를 받으면 쓰지 않는 iterator(워커 프로세스)를 하나 더 만들어
# Colab 에서 'can only test a child process' 경고를 쏟아냅니다 -> tqdm.notebook 을 씁니다
from tqdm.notebook import tqdm


def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)


SEED = 42
seed_everything(SEED)

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('사용 장치:', device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name())

## 하이퍼 파라미터

In [ ]:
params = {
    'epoch': 12,
    'batch_size': 32,
    'learning_rate': 1e-3,
    'valid_ratio': 0.15,   # 학습 2,906장 중 검증으로 떼어둘 비율
    'workers': 2,          # 본인 CPU 스레드의 절반 정도
    'num_show_img': 5,     # 데이터 확인용 표시 개수 (학습과 무관)
}

In [ ]:
# label 0 = 6공학관(6e), label 1 = IT관(it)
# ImageFolder는 폴더명을 알파벳순으로 정렬해 라벨을 매기므로 6e=0, it=1 이 됩니다.
class_names = {0: '6e', 1: 'it'}  # 그래프·이미지에 표시됨 -> 영문만 (Colab 에 한글 폰트 없음)
pt_file_name = 'model.pt'

## 모델 정의 (`model.py` 생성)

아래 셀은 `%%writefile` 매직으로 **`model.py` 파일을 생성**합니다. 이 파일은 나중에 모델 패키지에 그대로 들어가므로, 모델 구조는 반드시 이 셀에서 수정하세요. 수정 후에는 셀을 다시 실행해야 파일에 반영됩니다.

In [ ]:
%%writefile model.py
# 이 셀은 model.py 파일을 생성합니다 (모델 패키지에 포함되는 파일).
# 모델 구조를 수정하려면 이 셀을 수정한 뒤 다시 실행하세요.
import torch
import torch.nn as nn

# 전처리 상수 — 학습(노트북)과 추론(predict.py)이 반드시 같은 값을 써야 합니다.
# 여기 한 곳에서만 수정하세요. (평균/표준편차는 ImageNet 통계값)
IMG_SIZE = 224
NORM_MEAN = (0.485, 0.456, 0.406)
NORM_STD = (0.229, 0.224, 0.225)


def conv_block(in_ch, out_ch, pool=True):
    layers = [
        nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
    ]
    if pool:
        layers.append(nn.MaxPool2d(2))
    return nn.Sequential(*layers)


class SmallCNN(nn.Module):
    """약 40만 파라미터의 소형 CNN 베이스라인입니다.

    conv block 4개 + Global Average Pooling + Linear 구조로,
    돌리면 그럴듯한 점수가 나오는 출발점입니다. 채널 수, 블록 수, 증강,
    학습률 스케줄 등을 바꿔가며 개선해 보세요.
    """

    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            conv_block(3, 32),     # 224 -> 112
            conv_block(32, 64),    # 112 -> 56
            conv_block(64, 128),   # 56  -> 28
            conv_block(128, 256),  # 28  -> 14
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.2),
            nn.Linear(256, 1),  # 이진 분류: logit 1개 출력
        )

    def forward(self, x):
        x = self.features(x)
        x = self.head(x)
        return x  # sigmoid를 취하지 않은 raw logit을 반환 (loss에서 처리)


def build_model():
    """predict.py가 이 함수를 호출해 모델을 만듭니다. 구조를 바꿔도 이름은 유지하세요."""
    return SmallCNN()


In [ ]:
# model.py를 방금 다시 썼다면 커널이 이전 버전을 캐시하고 있을 수 있어 reload 합니다
import importlib
import model as model_module
importlib.reload(model_module)

model = model_module.build_model().to(device)
print(model)

### 파라미터 카운터 (20M 제한 확인)

앙상블을 하거나 사전학습 백본을 쓰는 경우 **모든 모델의 파라미터를 합산**한 값이 20,000,000 이하여야 합니다.

In [ ]:
PARAM_LIMIT = 20_000_000

n_params = sum(p.numel() for p in model.parameters())
print(f'총 파라미터 수: {n_params:,} / {PARAM_LIMIT:,} ({n_params / PARAM_LIMIT:.1%})')
assert n_params <= PARAM_LIMIT, '파라미터 20M 초과! 모델을 줄여주세요.'

## 데이터 불러오기 & 증강

- 학습(train) 데이터에만 증강을 적용하고, 검증(valid) 데이터는 증강 없이 평가합니다.
- 두 transform 모두 마지막에 채널별 **정규화(Normalize)** 를 적용합니다. 픽셀 분포를 평균 0 근처로 맞추면 학습이 안정됩니다.
- 증강은 자유입니다 (flip·crop·color·noise·mixup 등). 다양하게 실험해 보세요.
- **주의**: 최종 순위는 공개 리더보드에 보이지 않는 Private 250장으로 정합니다. 공개 점수에 맞춰 튜닝하기보다 밝기·색감·화각 변화에 강한 모델을 만드는 것이 최종 점수에 유리합니다.


In [ ]:
data_path = './train'

# 이미지 크기·정규화 상수는 model.py에 정의되어 있습니다 (predict.py와 공유).
IMG_SIZE = model_module.IMG_SIZE
NORM_MEAN = model_module.NORM_MEAN
NORM_STD = model_module.NORM_STD

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD),  # 채널별 정규화
])

# 검증·추론용 전처리 — predict.py의 transform과 반드시 동일해야 합니다
valid_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD),
])

# 같은 폴더를 transform만 다르게 두 번 로드한 뒤, 인덱스로 나눠 씁니다
full_train = datasets.ImageFolder(data_path, train_transform)
full_valid = datasets.ImageFolder(data_path, valid_transform)

print('클래스 → 라벨 매핑:', full_train.class_to_idx)
print('전체 이미지 수:', len(full_train))

### Train / Valid 분할

클래스 비율이 유지되도록 층화(stratified) 분할합니다. 검증 점수(log loss)가 리더보드 점수의 예측치 역할을 하므로, **검증 데이터를 학습에 섞지 않도록** 주의하세요.

In [ ]:
labels = [y for _, y in full_train.samples]
train_idx, valid_idx = train_test_split(
    list(range(len(full_train))),
    test_size=params['valid_ratio'],
    stratify=labels,
    random_state=SEED,
)

data_sets = {
    'train': Subset(full_train, train_idx),
    'valid': Subset(full_valid, valid_idx),
}
print(f"train: {len(data_sets['train'])}장 / valid: {len(data_sets['valid'])}장")

In [ ]:
g = torch.Generator()
g.manual_seed(SEED)

dataloaders = {
    'train': DataLoader(data_sets['train'], batch_size=params['batch_size'],
                        shuffle=True, num_workers=params['workers'], generator=g),
    'valid': DataLoader(data_sets['valid'], batch_size=params['batch_size'],
                        shuffle=False, num_workers=params['workers']),
}
print('배치 수 — train: %d / valid: %d' % (len(dataloaders['train']), len(dataloaders['valid'])))

### 데이터 확인

In [ ]:
def img_show(tensor, title=None):
    arr = tensor.numpy().transpose((1, 2, 0))
    arr = arr * np.array(NORM_STD) + np.array(NORM_MEAN)  # 정규화 되돌리기 (표시용)
    arr = np.clip(arr, 0, 1)
    plt.figure(figsize=(12, 4))
    plt.imshow(arr)
    plt.axis('off')
    if title is not None:
        plt.title(title)
    plt.show()


inputs, classes = next(iter(dataloaders['train']))
out = torchvision.utils.make_grid(inputs[:params['num_show_img']])
img_show(out, title=[class_names[int(c)] for c in classes[:params['num_show_img']]])

## 학습

손실함수는 `BCEWithLogitsLoss`를 사용합니다. 이 값이 곧 대회 지표인 **Log Loss와 동일**하므로, 검증 loss를 그대로 "내 예상 리더보드 점수"로 읽으면 됩니다.

검증 log loss가 가장 낮았던 epoch의 가중치를 `model.pt`로 저장합니다.

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    """optimizer가 주어지면 학습, 아니면 평가. (평균 loss, 정확도)를 반환."""
    training = optimizer is not None
    model.train() if training else model.eval()

    total_loss, total_correct, total_n = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for inputs, targets in tqdm(loader, desc='train' if training else 'valid', dynamic_ncols=True):
            inputs = inputs.to(device)
            targets = targets.float().unsqueeze(1).to(device)  # (B,) -> (B,1)

            logits = model(inputs)
            loss = criterion(logits, targets)

            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).float()
            total_loss += loss.item() * inputs.size(0)
            total_correct += (preds == targets).sum().item()
            total_n += inputs.size(0)

    return total_loss / total_n, total_correct / total_n


def train_model(model, dataloaders, criterion, optimizer, num_epochs):
    since = time.time()
    history = {'train_loss': [], 'train_acc': [], 'valid_loss': [], 'valid_acc': []}
    best_loss, best_epoch = float('inf'), -1

    for epoch in range(1, num_epochs + 1):
        print(f'\nEpoch {epoch}/{num_epochs}')
        tr_loss, tr_acc = run_epoch(model, dataloaders['train'], criterion, optimizer)
        va_loss, va_acc = run_epoch(model, dataloaders['valid'], criterion)

        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['valid_loss'].append(va_loss)
        history['valid_acc'].append(va_acc)
        print(f'train loss(=log loss) {tr_loss:.4f} / acc {tr_acc:.3f}')
        print(f'valid loss(=log loss) {va_loss:.4f} / acc {va_acc:.3f}')

        if va_loss < best_loss:
            best_loss, best_epoch = va_loss, epoch
            torch.save(model.state_dict(), pt_file_name)
            print(f'  ↳ best 갱신! {pt_file_name} 저장')

    elapsed = time.time() - since
    print(f'\n학습 완료 ({elapsed // 60:.0f}분 {elapsed % 60:.0f}초)')
    print(f'best: epoch {best_epoch}, valid log loss {best_loss:.4f}')
    return history, best_epoch, best_loss

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=params['learning_rate'])

In [ ]:
history, best_epoch, best_loss = train_model(
    model, dataloaders, criterion, optimizer, num_epochs=params['epoch'])

### 학습 곡선

train과 valid의 간격이 크게 벌어지면 과적합 신호입니다.

In [ ]:
epochs = range(1, len(history['train_loss']) + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, history['train_loss'], 'b-', label='train')
ax1.plot(epochs, history['valid_loss'], 'r-', label='valid')
ax1.plot(best_epoch, best_loss, 'ro')
ax1.set_xlabel('epoch'); ax1.set_ylabel('log loss'); ax1.legend(); ax1.grid()
ax1.set_title('Log Loss (competition metric)')

ax2.plot(epochs, history['train_acc'], 'b-', label='train')
ax2.plot(epochs, history['valid_acc'], 'r-', label='valid')
ax2.set_xlabel('epoch'); ax2.set_ylabel('accuracy'); ax2.legend(); ax2.grid()
ax2.set_title('Accuracy (reference only)')
plt.show()

## 테스트 추론 → `submission.csv` 생성

리더보드 제출 형식은 `id,prob` 입니다. `prob`은 **IT관(=1)일 확률**입니다. 제출은 **10분에 1회, 하루 최대 50회**입니다.

```
id,prob
f81d4fae-7dec-41d0-a765-00a0c91e6bf6,0.831250
c9b1f2e8-3a45-4d67-b012-5e6f7a8b9c0d,0.024113
```

> 채점 시 확률은 [1e-6, 1-1e-6]로 클리핑되지만, 0이나 1에 극단적으로 붙은 확신은 한 장만 틀려도 log loss를 크게 망가뜨립니다. 모델이 과확신하지 않는지 확인해 보세요.

In [ ]:
class TestImageDataset(Dataset):
    def __init__(self, files, transform):
        self.files = files
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, index):
        file = self.files[index]
        img = Image.open(file).convert('RGB')
        return self.transform(img), os.path.basename(file)


import glob
test_files = sorted(glob.glob('./test/*'))
print('테스트 이미지 수:', len(test_files))

test_loader = DataLoader(
    TestImageDataset(test_files, valid_transform),
    batch_size=params['batch_size'], shuffle=False)

In [ ]:
# 검증 성적이 가장 좋았던 시점의 가중치를 불러와 추론합니다
best_model = model_module.build_model()
best_model.load_state_dict(torch.load(pt_file_name, map_location=device))
best_model = best_model.to(device)
best_model.eval()

rows = ['id,prob']
with torch.no_grad():
    for inputs, filenames in tqdm(test_loader, desc='inference', dynamic_ncols=True):
        probs = torch.sigmoid(best_model(inputs.to(device))).squeeze(1)
        for name, p in zip(filenames, probs):
            rows.append(f'{os.path.splitext(name)[0]},{p.item():.6f}')

with open('submission.csv', 'w') as f:
    f.write('\n'.join(rows) + '\n')

print('submission.csv 생성 완료 — 리더보드에 이 파일을 제출하세요.')
print('\n'.join(rows[:6]))

## 최종 파일 생성 (예시: `predict.py` 패키지)

마감 후 운영진이 **모든 참가자의 최종 파일**을 받아 **Colab CPU** 환경에서 직접 실행합니다.

1. 공개 리더보드(Public) 점수가 재현되는지 확인
2. 런타임(테스트 500장 10분 이내)과 파라미터 수(20M 이하) 확인
3. 통과한 참가자만 Private 250장 점수로 최종 순위 산정

**제출 형식은 자유**입니다. 운영진이 실행할 수 있고 가중치가 포함되어 있으면 됩니다. 아래는 이 노트북이 만드는 예시 형식이며, 이렇게 내면 다음 명령 한 줄로 검증됩니다.

```shell
python predict.py --input <이미지 폴더> --output pred.csv
```

패키지 구성:
```
📦submission_package.zip
 ├─📜model.py     # 모델 구조 정의 (build_model 함수 필수)
 ├─📜predict.py   # 추론 진입점 (--input/--output 인터페이스 유지)
 └─📜model.pt     # 학습된 가중치 (사전학습 가중치 사용 시에도 여기에 포함)
```

In [ ]:
%%writefile predict.py
# 이 셀은 predict.py 파일을 생성합니다 (모델 패키지에 포함되는 파일).
# 운영진은 Colab CPU 에서 이 파일을 실행해 공개 점수 재현·런타임·파라미터 수를 확인합니다:
#   python predict.py --input <이미지 폴더> --output pred.csv
#
# 전처리·TTA 등 추론 로직을 바꾸고 싶다면 이 파일을 수정하면 됩니다.
# 제출 형식은 자유지만, 출력 CSV 는 id,prob 형식이어야 채점할 수 있습니다.
import argparse
import glob
import os

import torch
from PIL import Image
from torchvision import transforms

from model import IMG_SIZE, NORM_MEAN, NORM_STD, build_model

PARAM_LIMIT = 20_000_000


def load_images(input_dir):
    # 확장자를 소문자로 비교 -> OS 에 상관없이 한 파일은 한 번만 잡힌다
    exts = (".jpg", ".jpeg", ".png")
    return sorted(f for f in glob.glob(os.path.join(input_dir, "*"))
                  if f.lower().endswith(exts))


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--input", required=True, help="테스트 이미지 폴더")
    parser.add_argument("--output", required=True, help="출력 CSV 경로")
    parser.add_argument("--weights", default="model.pt", help="모델 가중치 파일")
    args = parser.parse_args()

    device = torch.device("cpu")  # 검증은 Colab CPU 기준입니다

    weights_path = os.path.join(os.path.dirname(os.path.abspath(__file__)), args.weights)
    model = build_model()
    model.load_state_dict(torch.load(weights_path, map_location=device))
    model.to(device)
    model.eval()

    n_params = sum(p.numel() for p in model.parameters())
    print(f"파라미터 수: {n_params:,} (제한 {PARAM_LIMIT:,})")
    assert n_params <= PARAM_LIMIT, "파라미터 20M 초과! 실격 대상입니다."

    # 학습 때의 valid_transform과 동일한 전처리 (다르면 채점 때 점수가 무너집니다)
    transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(NORM_MEAN, NORM_STD),
    ])

    files = load_images(args.input)
    assert len(files) > 0, f"{args.input} 에서 이미지를 찾지 못했습니다."

    rows = ["id,prob"]
    with torch.no_grad():
        for i in range(0, len(files), 32):
            batch_files = files[i:i + 32]
            batch = torch.stack([
                transform(Image.open(f).convert("RGB")) for f in batch_files
            ]).to(device)
            probs = torch.sigmoid(model(batch)).squeeze(1)
            for f, p in zip(batch_files, probs):
                rows.append(f"{os.path.splitext(os.path.basename(f))[0]},{p.item():.6f}")

    with open(args.output, "w") as fp:
        fp.write("\n".join(rows) + "\n")
    print(f"{args.output} 저장 완료 ({len(files)}장)")


if __name__ == "__main__":
    main()


In [ ]:
# 패키지 폴더 구성 및 압축
if os.path.exists('package'):
    shutil.rmtree('package')
os.makedirs('package')
for f in ['model.py', 'predict.py', pt_file_name]:
    shutil.copy(f, 'package/')

zip_path = shutil.make_archive('submission_package', 'zip', 'package')
print('패키지 생성 완료:', zip_path)
print('내용물:', sorted(os.listdir('package')))

### 패키지 자가 검증 (제출 전 필수!)

운영진 검증과 같은 방식으로, 패키지 폴더 안에서 `predict.py`를 CPU로 실행해 봅니다. 여기서 에러가 나면 검증도 실패합니다. Colab 런타임을 CPU로 두고 실행하면 10분 기준도 함께 확인할 수 있습니다.

In [ ]:
import sys
result = subprocess.run(
    [sys.executable, 'predict.py', '--input', '../test', '--output', '../package_check.csv'],
    cwd='package', capture_output=True, text=True)
print(result.stdout)
print(result.stderr)
assert result.returncode == 0, 'predict.py 실행 실패! 패키지를 점검하세요.'

# 노트북 추론 결과와 패키지 추론 결과가 (허용 오차 내에서) 일치하는지 확인
# GPU로 학습·추론한 경우 CPU 추론과 부동소수점 미세 차이가 나는 것은 정상입니다
def read_probs(path):
    with open(path) as f:
        next(f)  # header
        return {line.split(',')[0]: float(line.split(',')[1]) for line in f if line.strip()}

nb_probs = read_probs('submission.csv')
pkg_probs = read_probs('package_check.csv')
assert nb_probs.keys() == pkg_probs.keys(), '파일 목록이 다릅니다!'
max_diff = max(abs(nb_probs[k] - pkg_probs[k]) for k in nb_probs)
print(f'노트북 vs 패키지 확률 최대 차이: {max_diff:.2e}')
assert max_diff < 1e-3, '차이가 너무 큽니다 — 전처리나 가중치가 어긋났는지 확인하세요.'
print('패키지 자가 검증 통과 ✓')

## 제출 전 체크리스트

- [ ] `submission.csv`가 `id,prob` 형식이고 테스트 500장 전부를 포함하는가?
- [ ] 파라미터 카운터 셀에서 **20M 이하**를 확인했는가? (앙상블·사전학습 백본 합산)
- [ ] 패키지 자가 검증 셀이 에러 없이 통과했는가? (Colab CPU 에서 500장 10분 이내)
- [ ] `predict.py`가 네트워크 없이 동작하는가? (다운로드 코드 금지, 가중치는 패키지에 동봉)
- [ ] 테스트 이미지를 학습에 사용하지 않았는가? (pseudo-labeling 포함 금지)
- [ ] 외부 이미지를 학습에 사용하지 않았는가?

## 개선 아이디어

- 채널 수·블록 수 조정, 학습률 스케줄러(CosineAnnealing 등) 추가
- 20M 이내의 사전학습 백본(MobileNetV3, EfficientNet-B0 등) fine-tuning — 가중치를 패키지에 포함할 것
- 증강 다양화: 밝기·색감·회전·노이즈 — 처음 보는 촬영 조건에 대한 일반화
- 검증 전략 개선: k-fold 교차검증으로 점수 추정의 신뢰도 높이기
- 과확신 완화: label smoothing, 확률 클리핑, 온도 조절(temperature scaling)
